# IT Incidents — Similarity Model

This notebook trains the model the Django application uses to detect **similar — potentially duplicate — incidents**: when someone reports a new incident, the app suggests the open incidents that look most like it, so the same problem is not registered twice.

The model is deliberately simple: a TF-IDF vectorizer (`scikit-learn`) turns the text of each incident (title, description and affected equipment) into a sparse vector, and the similarity between two incidents is the cosine of the angle between their vectors. No labels are needed, so it works with whatever incidents exist.

Like the analysis and dashboard notebooks, it reads directly from the project's SQLite database (`src/db.sqlite3`) through a read-only connection, without importing Django or its ORM. The fitted vectorizer is serialized with `joblib` to `ml/similarity.joblib`; the web application loads that file only for inference and never retrains it per request.

Run this notebook from the repository root (the same working directory as `analysis/analysis.ipynb`). With an empty database, populate it first with `make seed`.

## Load incidents

All incidents are loaded — closed and archived ones included — because the vectorizer only learns the vocabulary and the weight (IDF) of each term. Which incidents are offered as candidates (e.g. only open ones) is a decision taken at inference time, not here.

In [ ]:
import sqlite3
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DB_PATH = Path("src/db.sqlite3").resolve()
MODEL_PATH = Path("ml/similarity.joblib")

connection = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
incidents = pd.read_sql_query(
    "SELECT id, title, description, equipment, priority, status, is_archived FROM incidents_incident",
    connection,
    index_col="id",
)
connection.close()

print(f"{len(incidents)} incidents loaded")
incidents.head()

## Build the text corpus

Each incident becomes a single string made of its title, description and equipment. The equipment name matters: "Printer-2F" and "Printer-Lobby" are different problems even if the wording is identical.

`build_text` is the contract between training and inference: the application must build the text of a new incident exactly the same way before transforming it with the saved vectorizer, so this function will live in `ml/similarity.py` and be imported from both places.

In [ ]:
from ml.similarity import build_text

incidents["text"] = [
    build_text(row.title, row.description, row.equipment) for row in incidents.itertuples()
]
incidents[["title", "text"]].head()

## Fit the TF-IDF vectorizer

- `stop_words="english"`: the application's UI and data are in English, so common words ("the", "is", "not") are dropped.
- `ngram_range=(1, 2)`: besides single words, pairs of consecutive words become terms, so "paper jam" or "no signal" count as one distinctive feature.
- `sublinear_tf=True`: a word repeated five times in a description is not five times more important; the term frequency is damped with `1 + log(tf)`.

Fitting learns the vocabulary and the IDF of each term — how rare it is across all incidents. Rare terms get a higher weight than terms that appear everywhere.

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
tfidf_matrix = vectorizer.fit_transform(incidents["text"])

print(f"Vocabulary size: {len(vectorizer.vocabulary_)} terms")
print(f"Matrix shape: {tfidf_matrix.shape} (incidents x terms)")

The terms with the lowest IDF are the ones shared by most incidents; they carry the least weight when comparing two texts.

In [ ]:
idf = pd.Series(vectorizer.idf_, index=vectorizer.get_feature_names_out(), name="idf")
idf.sort_values().head(15).to_frame()

## Explore similarities

`cosine_similarity` compares every incident against every other one. `most_similar` mirrors what the application will do for a new incident: transform its text with the fitted vectorizer, score it against the existing incidents and return the best matches.

In [ ]:
similarity_matrix = cosine_similarity(tfidf_matrix)


def most_similar(query: str, k: int = 5, exclude_id: int | None = None) -> pd.DataFrame:
    """Return the ``k`` incidents most similar to ``query``, with their cosine score."""
    scores = cosine_similarity(vectorizer.transform([query]), tfidf_matrix)[0]
    ranked = incidents[["title", "equipment", "status"]].assign(score=scores)
    if exclude_id is not None:
        ranked = ranked.drop(index=exclude_id)
    return ranked.sort_values("score", ascending=False).head(k)

For a few incidents picked at random, the best matches should describe the same kind of problem (same family of wording, same or related equipment).

In [ ]:
for incident_id in incidents.sample(3, random_state=0).index:
    row = incidents.loc[incident_id]
    print(f"#{incident_id} — {row.title} [{row.equipment}]")
    display(most_similar(row.text, k=5, exclude_id=incident_id))

## Choosing a threshold

Returning the top-k matches is not enough: an incident with no real duplicate still has a "best" match, only with a low score. The application will therefore ignore matches below a minimum score.

The left histogram shows every pairwise similarity; most pairs are unrelated and score near zero. The right one shows, for each incident, the score of its best match against the rest.

In [ ]:
pairwise = similarity_matrix.copy()
np.fill_diagonal(pairwise, np.nan)
best_match = pd.Series(np.nanmax(pairwise, axis=1), index=incidents.index, name="best_match")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(pairwise[~np.isnan(pairwise)], bins=40, color="steelblue")
axes[0].set_title("All pairwise similarities")
axes[0].set_xlabel("cosine similarity")
axes[1].hist(best_match, bins=20, color="darkorange")
axes[1].set_title("Best match per incident")
axes[1].set_xlabel("cosine similarity")
plt.tight_layout()
plt.show()

best_match.describe().to_frame()

In [ ]:
for threshold in (0.1, 0.2, 0.3, 0.4, 0.5):
    share = np.nanmean(pairwise >= threshold)
    print(f"threshold {threshold:.1f}: {share:5.1%} of all pairs would be flagged as similar")

Unrelated pairs pile up near zero, while incidents about the same problem typically score above 0.4 — the median best match is well above that. Short texts can still reach roughly 0.2 by sharing a single generic word ("floor", "minutes"), as the last example below shows, so the cut must sit above that noise.

A minimum score of **0.3** keeps the matches that share a distinctive term or two (same equipment, "paper jam", "no signal") and discards the coincidences. It is the default the inference module will use; tune it here if the real data behaves differently.

In [ ]:
MIN_SCORE = 0.3

## Try a new incident

This is the scenario the application faces: a user is typing a new incident and the app must decide whether something similar is already registered.

In [ ]:
new_incident = build_text(
    "Printer on the 2nd floor is not printing",
    "Jobs are sent but nothing comes out and the display shows a paper jam.",
    "Printer-2F",
)
results = most_similar(new_incident, k=5)
results[results["score"] >= MIN_SCORE]

A problem nobody has reported before should produce no suggestions at all. Its best matches only share a generic word with the query and stay below the threshold, so the filtered result is empty.

In [ ]:
unrelated = build_text(
    "Coffee machine broken",
    "The coffee machine in the kitchen leaks water on the floor.",
    "Kitchen",
)
results = most_similar(unrelated, k=5)
display(results)
results[results["score"] >= MIN_SCORE]

## Save the model

Only the fitted vectorizer is serialized. The TF-IDF matrix of the existing incidents is *not* saved: incidents are created and closed all the time, so the application recomputes the candidates' vectors at inference time with the same vectorizer. Re-run this notebook to refresh the vocabulary when the data has changed substantially.

In [ ]:
joblib.dump(vectorizer, MODEL_PATH)
print(f"Saved {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1024:.1f} KiB)")